In [1]:
import os
import numpy as np
import pandas as pd
import xrayutilities as xu
from xrayutilities.io.imagereader import get_tiff
from xrayutilities.gridder3d import FuzzyGridder3D

# Optional: for VTK export
import vtk
from vtk.util import numpy_support


In [11]:
from xrayutilities.io.spec import SPECFile

# Open the SPEC file.
spec =SPECFile('/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23')

# Select scan number 10 (or the desired one).
scan = spec[24]

spec_lines = open("/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23").readlines()

hlk_data = []

for line in spec_lines:
    if line.startswith("#L"):
        headers = line.strip().split()[1:]
        h_idx = headers.index('H')
        k_idx = headers.index('K')
        l_idx = headers.index('L')
    elif not line.startswith("#") and line.strip():
        parts = line.strip().split()
        h = float(parts[h_idx])
        k = float(parts[k_idx])
        l = float(parts[l_idx])
        hlk_data.append([h, k, l])

import numpy as np
hlk_array = np.array(hlk_data)  # shape (N, 3)

In [ ]:
import xrayutilities as xu
import os

In [7]:
import xrayutilities as xu
import os
 
# open spec file or use open SPECfile instance
try: s
except NameError:
     s = xu.io.SPECFile("/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23")
 
 # method (1)
s.scan10.ReadData()
print(len(s.scan10.header))  # print header information
scan10data = s.scan10.data
print(scan10data[1][2])

# method (2)
# h5file = os.path.join("h5dir", "h5file.h5")
# s.Save2HDF5(h5file) # save content of SPEC file to HDF5 file
# # read data from HDF5 file
# [angle1, angle2], scan10data = xu.io.geth5_scan(h5file, [10],
#                                                 "motorname1",
#                                                 "motorname2")

XU.io.SPECScan.ReadData: scan_10: 21 25 25
21
4.0314703


In [13]:
print("H, K, L values extracted from the SPEC file:")
print(hlk_array.shape)

H, K, L values extracted from the SPEC file:
(606, 3)


In [3]:
with open('/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23', 'r') as f:
    contents = f.read()
print(contents)

#F setup_6oct23
#E 1696599326
#D Fri Oct 06 09:35:26 2023
#C fourc  User = xf04id

#O0 VTTH  VTH  Chi  Phi  pathf  DCM_Chi2  DCM_th2f  DCM_chi2f
#O1 MD_StgY  MD_TX  BPM3y  BPM3x  DCM_BRAGG  MON_Y  MON_Z  VFM_PITCH
#O2 DCM_CB  MD_TH  MD_TTH  MD_PHI  MD_CHI  MD_ATH  MD_ATTH  MD_ty1
#O3 MD_ty2  MD_ty3  JJtop  JJin  SSA_XG  SSA_XC  SSA_YG  SSA_YC
#O4 VFM_BEND  JJout  JJbot  Sample_X  Sample_Y  Sample_Z  VCS_b  VCS_i
#O5 VCS_o  VCS_t  VTS_b  VTS_i  VTS_o  VTS_t  path  DCM_th2
#O6 pachi  patth  dpp1x  dpp1y  dpp1yaw  dpp1chi  dpp1th  dpp1tth
#O7 hth  htth  dpp2x  dpp2y  dpp2yaw  dpp2chi  dpp2th  dpp2tth
#O8 dpp2sy  dpp1sy  spz  dhrmx  dhrmp  dhrmy  dhrmfr  dhrmfp
#O9 vfmy  iuvgap  Energy  
#o0 tth th chi phi pathf dcm_ch2 th2f chi2f
#o1 stgy mdtx bpm3y bpm3x mono mon_y mon_z vfm_pit
#o2 dcm_cb md_th md_tth md_phi md_chi md_ath md_atth md_ty1
#o3 md_ty2 md_ty3 jjt jji ssaxg ssaxc ssayg ssayc
#o4 vfmbend jjo jjb samx samy samz ds_b ds_i
#o5 ds_o ds_t gs_b gs_i gs_o gs_t path dcm_th2
#o6 pachi 

In [ ]:

# -----------------------------------------------------------------------------
# 0. User switches: SPEC vs Manual metadata
# -----------------------------------------------------------------------------
use_spec = False             # ← set to True to parse a SPEC file
spec_file   = '/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23.spec' # your SPEC file
scan_number = 10             # which scan in that SPEC

# If not using SPEC, put your metadata in one of two ways:
#  a) In a CSV file with columns: filename, omega, chi, phi, twotheta
#     meta = pd.read_csv('metadata.csv')
#  b) Directly in Python:
#     meta = pd.DataFrame([
#         {'filename':'img001.tiff','omega':0.0,'chi':0.0,'phi':0.0,'twotheta':30.0},
#         {'filename':'img002.tiff','omega':0.1,'chi':0.0,'phi':0.0,'twotheta':30.1},
#         # ...
#     ])
if not use_spec:
    # Example: manual entry
    meta = pd.DataFrame([
        {'filename':'img001.tiff','omega':0.0,'chi':0.0,'phi':0.0,'twotheta':30.0},
        {'filename':'img002.tiff','omega':0.1,'chi':0.0,'phi':0.0,'twotheta':30.1},
        # add as many frames as you have...
    ])

# -----------------------------------------------------------------------------
# 1. Define crystal & diffractometer geometry
# -----------------------------------------------------------------------------
Si   = xu.materials.Si
Q1   = Si.Q(1, 0, 0)     # in-plane axis
Q2   = Si.Q(0, 0, 1)     # out-of-plane axis
hxrd = xu.HXRD(Q1, Q2)   # builds a 4-circle QConversion
qconv = hxrd._qconv     # use for area detector → (h,k,l)



In [ ]:
# -----------------------------------------------------------------------------
# 2. Prepare your gridder
# -----------------------------------------------------------------------------
h_bins, k_bins, l_bins = 200, 200, 200
gridder = FuzzyGridder3D(h_bins, k_bins, l_bins)  # finite-volume binning

# -----------------------------------------------------------------------------
# 3. Loop: load each frame, convert & bin
# -----------------------------------------------------------------------------
if use_spec:
    # parse SPEC -> returns angles + image dir
    from xrayutilities.io import get_average_RSM
    # This helper will both parse SPEC and populate the gridder internally
    # but if you prefer full manual control, you can extract the metadata via
    # xu.io.specfile.SPECReader and then follow the TIFF loop below.
    grid3D = get_average_RSM(
        h_bins, k_bins, l_bins,
        qconv,
        specfile=spec_file,
        specnr=scan_number,
        datadir='.',
        keepdir=0, roi=None, nav=(1,1), filterfunc=None
    )
    h_axis, k_axis, l_axis = grid3D.xaxis, grid3D.yaxis, grid3D.zaxis
    rsm = grid3D.data

else:
    # manual TIFF + metadata loop
    tiff_folder = 'tiff_folder'
    for _, row in meta.iterrows():
        # read TIFF
        img = get_tiff(os.path.join(tiff_folder, row.filename)).data

        # convert to h,k,l,I arrays
        h, k, l, I = qconv.area(
            img,
            row.omega, row.chi, row.phi, row.twotheta,
            roi=None, normalize=False
        )

        # bin into grid
        gridder(h.ravel(), k.ravel(), l.ravel(), I.ravel())

    # extract axes & data
    h_axis, k_axis, l_axis = gridder.xaxis, gridder.yaxis, gridder.zaxis
    rsm = gridder.data

In [ ]:
# -----------------------------------------------------------------------------
# 4. Export to VTK (.vti) for ParaView / Mayavi
# -----------------------------------------------------------------------------
# Build a vtkImageData (uniform grid) and attach 'Intensity'
H, K, L = rsm.shape
img = vtk.vtkImageData()
img.SetDimensions(H, K, L)

# assume uniform spacing
dh = float(h_axis[1] - h_axis[0])
dk = float(k_axis[1] - k_axis[0])
dl = float(l_axis[1] - l_axis[0])
img.SetSpacing(dh, dk, dl)
img.SetOrigin(float(h_axis[0]), float(k_axis[0]), float(l_axis[0]))

# flatten in Fortran order (VTK expects x-fastest)
flat_int = np.ravel(rsm, order='F').astype(np.float32)
vtk_arr = numpy_support.numpy_to_vtk(
    num_array=flat_int,
    deep=True,
    array_type=vtk.VTK_FLOAT
)
vtk_arr.SetName("Intensity")
img.GetPointData().AddArray(vtk_arr)
img.GetPointData().SetActiveScalars("Intensity")

# write .vti
writer = vtk.vtkXMLImageDataWriter()
writer.SetFileName("rsm_map.vti")
writer.SetInputData(img)
if writer.Write():
    print("Wrote VTK volume: rsm_map.vti")
else:
    raise RuntimeError("Failed to write VTK file")